[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NASA-Gradient-Descent-Farmers/NASA-warm-up/blob/sant-calibration/Notebooks/yield_panel.ipynb)

# Municipal yield and failure panel: can NASA weather beat "average + trend"?

* **Author:** Santiago (Gradient Descent Farmers) · **Branch:** `sant-calibration`
* **Code:** `src/yield_panel/` · **Tests:** `tests/test_yield_panel.py`

**Question.** For rainfed spring-summer crops in Mexico, does each season's NASA weather explain how a municipality's results move from year to year, better than the municipality's own average plus a trend?

**Unit.** One row = municipality × year × crop (SIAP, *Primavera-Verano, temporal*).

**Two targets.**
* `yield_t_ha`: production ÷ harvested area. How much the fields that were harvested produced.
* `failure_rate`: lost area (*siniestrada*) ÷ sown area. The share of the planting that was lost.

Production per sown hectare ≈ (1 − failure) × yield.

**Inputs.** Season weather from NASA POWER (one request per ~55 × 65 km grid cell, shared by the municipalities inside it), GLDAS soil texture, and INEGI location and altitude.

**The bar.** `mean_trend`: each municipality's own average plus one trend shared by all. Isaac's single-site Texcoco models did not beat this kind of baseline. The panel gives far more rows, and most importantly many more *independent seasons*, because weather varies across cells as well as across years.

**How to read a win.** The models use the *realized* season's weather. Beating the bar means the model has learned how yield responds to weather, which is what is needed to run a municipality through 20+ past seasons and get a risk profile (section 6). It is not a pre-season forecast: nobody knows the season's weather at sowing.

## 0. Setup

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/NASA-Gradient-Descent-Farmers/NASA-warm-up.git"
BRANCH = "sant-calibration"

if "google.colab" in sys.modules:
    if not Path("/content/NASA-warm-up").exists():
        !git clone -q --branch {BRANCH} {REPO_URL} /content/NASA-warm-up
    !pip install -q "earthaccess>=0.11"
    ROOT = Path("/content/NASA-warm-up")
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "yield_panel").is_dir())

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Repo root:", ROOT)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.yield_panel import build, models as pm, weather

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

### Configuration

Start with central Mexico (11 states, a few hundred POWER cells). `STATES = None` uses the whole country: more data, more variety of climates, longer download.

**First run:** ~25 SIAP national files (tens of MB each) and one POWER request per grid cell, both cached under `data/`. Expect roughly 20-60 minutes for central Mexico, depending on POWER's response times. Later runs read the cache.

In Colab, `data/` lives in the runtime and disappears when it resets. To keep the cache, mount Google Drive and point `CACHE_TO_DRIVE` at a folder.

In [ ]:
STATES = build.CENTRAL_STATES      # INEGI codes; None = all of Mexico
CROPS = ["maize", "bean", "sorghum", "wheat", "barley", "oats"]
YEARS = range(2003, 2026)          # SIAP municipal series starts in 2003
TEST_YEARS = range(2014, 2026)     # each year is predicted from the years before it
MIN_SOWN_HA = 50                   # drop tiny plantings (noisy)
MIN_YEARS = 10                     # each municipality-crop series needs this many years
PANEL_PATH = Path("data/yield_panel/panel_central.csv")
CACHE_TO_DRIVE = None              # e.g. "/content/drive/MyDrive/space_apps_cache"

if CACHE_TO_DRIVE and "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CACHE_TO_DRIVE).mkdir(parents=True, exist_ok=True)
    if not Path("data").is_symlink():
        !rm -rf data && ln -s {CACHE_TO_DRIVE} data

### Sanity check: one POWER request per grid cell

`weather.py` snaps each municipality to the centre of its POWER cell so that municipalities in the same cell share one request. This compares the API at the municipality point and at the snapped centre for three municipalities. `max_abs_diff` should be 0. If it isn't, set `weather.snap = lambda lat, lon: (lat, lon)` to request every municipality separately (slower but always correct).

In [ ]:
print(weather.check_snapping(weather.load_municipalities(), n=3))

## 1. Build the panel

In [ ]:
if PANEL_PATH.exists():
    panel = pd.read_csv(PANEL_PATH)
    print("Loaded cached panel:", PANEL_PATH)
else:
    panel, info = build.build_panel(YEARS, CROPS, STATES, min_sown_ha=MIN_SOWN_HA, min_years=MIN_YEARS, workers=4)
    PANEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    panel.to_csv(PANEL_PATH, index=False)
    for key, value in info.items():
        print(f"{key}: {value}")

print(panel.shape)
panel.head()

## 2. Look at the data first

In [ ]:
summary = panel.groupby("crop").agg(
    rows=("year", "size"),
    municipalities=("mun_id", lambda s: panel.loc[s.index, ["state_id", "mun_id"]].drop_duplicates().shape[0]),
    sown_kha_per_year=("sown_ha", lambda s: s.sum() / panel["year"].nunique() / 1000),
    median_yield=("yield_t_ha", "median"),
    mean_failure=("failure_rate", "mean"),
    share_years_failure_over_30pct=("failure_rate", lambda s: (s > 0.3).mean()),
).round(2).sort_values("rows", ascending=False)
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
main_crops = summary.index[:4]

for crop in main_crops:
    d = panel[panel.crop == crop]
    by_year = d.groupby("year").apply(lambda g: np.average(g.yield_t_ha.dropna(), weights=g.loc[g.yield_t_ha.notna(), "harvested_ha"]),
                                      include_groups=False)
    axes[0].plot(by_year.index, by_year.values, "o-", ms=3, label=crop)
    fail = d.groupby("year").apply(lambda g: g.lost_ha.sum() / g.sown_ha.sum(), include_groups=False)
    axes[1].plot(fail.index, fail.values, "o-", ms=3, label=crop)
axes[0].set(title="Area-weighted yield (harvested)", ylabel="t/ha")
axes[1].set(title="Share of sown area lost", ylabel="fraction")
axes[0].legend()

maize = panel[panel.crop == "maize"].groupby(["state_id", "mun_id"]).agg(lat=("lat", "first"), lon=("lon", "first"),
                                                                         y=("yield_t_ha", "mean"))
sc = axes[2].scatter(maize.lon, maize.lat, c=maize.y, s=10, cmap="viridis")
plt.colorbar(sc, ax=axes[2], label="t/ha")
axes[2].set(title="Mean rainfed maize yield by municipality", xlabel="lon", ylabel="lat")
fig.tight_layout()
plt.show()

Things to check before modelling: years where every crop fails together (a national drought, e.g. 2011), sudden level shifts that look like reporting changes, and municipalities whose failure rate is always 0 (some states report losses unevenly).

## 3. Features

All `weather.WEATHER_FEATURES` (rain totals by period, rain before the season, wet days, heaviest day, longest dry spell, rain onset, degree days, heat days, late frost, humidity, radiation), plus squared terms for season rain, degree days and max temperature so effects can bend. The boosted model also sees location, altitude, soil and each municipality's typical level.

In [ ]:
data, FEATURES = pm.add_terms(panel, weather.WEATHER_FEATURES)
STATIC = ["lat", "lon", "alt_m", "sand", "clay", "porosity"]
MODELS = pm.make_models(FEATURES, STATIC)
TARGETS = ["yield_t_ha", "failure_rate"]
print(len(FEATURES), "weather terms:", FEATURES)

## 4. Rolling evaluation: does weather beat the bar?

For every test year t, each model is fitted on years before t only and predicts t (expanding window). All models are scored on the same rows. `skill_vs_bar` = 1 − RMSE / RMSE(`mean_trend`): above 0 beats the bar.

The bootstrap resamples whole **test years**, because all municipalities in a year share related weather shocks: years, not rows, are the independent evidence. A gain whose interval stays above 0 is a real result.

In [ ]:
results, predictions, intervals = [], {}, []
for crop in summary.index:
    d = data[data.crop == crop]
    for target in TARGETS:
        preds = pm.rolling_evaluation(d, target, MODELS, TEST_YEARS, min_train_years=8)
        if preds.empty:
            continue
        predictions[(crop, target)] = preds
        table = pm.score(preds, MODELS).assign(crop=crop, target=target)
        results.append(table.reset_index(names="model"))
        for model in ["fe_weather", "gbm_weather"]:
            intervals.append({"crop": crop, "target": target, **pm.bootstrap_skill(preds, model, draws=1000)})
        print(f"done: {crop} / {target} ({len(preds)} rows)", flush=True)

results = pd.concat(results, ignore_index=True)
skill = results.pivot_table(index=["crop", "target"], columns="model", values="skill_vs_bar")[list(MODELS)]
skill.round(3)

In [ ]:
pd.DataFrame(intervals).round(4)

In [ ]:
crop, target = "maize", "yield_t_ha"
by_year = pm.score_by_year(predictions[(crop, target)], MODELS)
ax = by_year.plot(marker="o", figsize=(11, 3.8), title=f"{crop} / {target}: RMSE per test year (lower is better)")
ax.set_ylabel("RMSE")
plt.show()

## 5. What the fixed-effects model learned

`effect_per_sd` is the change in the target for a one-standard-deviation move in a feature *within* a municipality (its own year-to-year variation). This is where to check agronomic sense: drier canícula → lower yield and more failure, later onset → worse, more heat days → worse. A large effect with the "wrong" sign is a reason to dig into the data before trusting the model.

In [ ]:
def fe_effects(crop, target):
    d = data[data.crop == crop].dropna(subset=[target])
    fit = pm.FixedEffects(["year", *FEATURES]).fit(d, target)
    within_sd = (d[FEATURES] - d.groupby(pm.UNIT)[FEATURES].transform("mean")).std()
    out = pd.DataFrame({"coef": fit.coef_.drop("year"), "effect_per_sd": fit.coef_.drop("year") * within_sd})
    return out.reindex(out["effect_per_sd"].abs().sort_values(ascending=False).index)

pd.concat({t: fe_effects("maize", t) for t in TARGETS}, axis=1).round(4)

## 6. From model to decision support: a municipality's risk profile

Fit on all years, hold the trend at the latest year, and replay every past season's weather. Each crop gets a distribution instead of a single number: median, bad-year (10th percentile) yield, chance of losing more than 30% of the sown area, and expected production per sown hectare.

Use the model that won section 4 for each target (`fe_weather` is the easier one to explain).

In [ ]:
UNIT_NAME = "Celaya"                 # any municipality in the panel
MODEL_NAME = {"yield_t_ha": "fe_weather", "failure_rate": "fe_weather"}

unit_row = panel[panel.municipality == UNIT_NAME][["state_id", "mun_id"]].drop_duplicates()
if unit_row.empty:
    raise SystemExit(f"{UNIT_NAME} has no rainfed series in the panel; pick another municipality")
UNIT = tuple(unit_row.iloc[0])
season_weather = panel.drop_duplicates(["state_id", "mun_id", "year"])  # weather rows per municipality-year

profiles = []
for crop in sorted(data.loc[(data.state_id == UNIT[0]) & (data.mun_id == UNIT[1]), "crop"].unique()):
    d = data[data.crop == crop]
    out = {}
    for target in TARGETS:
        model = MODELS[MODEL_NAME[target]]().fit(d, target)
        sim = pm.risk_profile(model, d, season_weather, UNIT, max(YEARS), weather.WEATHER_FEATURES)
        out[target] = pm.clip_target(target, sim.set_index("weather_year")["prediction"])
    sim = pd.DataFrame(out)
    sim["per_sown_ha"] = (1 - sim.failure_rate) * sim.yield_t_ha
    profiles.append({
        "crop": crop, "seasons": len(sim),
        "median_yield": sim.yield_t_ha.median(), "p10_yield": sim.yield_t_ha.quantile(0.1),
        "mean_failure": sim.failure_rate.mean(), "p_failure_over_30pct": (sim.failure_rate > 0.3).mean(),
        "median_per_sown_ha": sim.per_sown_ha.median(), "p10_per_sown_ha": sim.per_sown_ha.quantile(0.1),
    })
print(f"{UNIT_NAME} {UNIT}: rainfed spring-summer, trend at {max(YEARS)}, weather of {min(YEARS)}-{max(YEARS)}")
pd.DataFrame(profiles).set_index("crop").round(2)

Yields are on SIAP's reported basis per crop, so compare crops by risk (failure chance, P10 relative to median), and by income only after adding prices (`precio`/`valorproduccion` in the SIAP files).

## 7. Notes and next steps

* **If the bar wins:** check the inputs before the model. POWER cells are coarse; try fewer features, state-specific trends, or restricting to municipalities whose farmland sits close to the cell's elevation.
* **Irrigation leaks in:** SIAP's *temporal* label is municipal; some "rainfed" areas get supplemental water. Municipalities with low weather sensitivity may be telling you that.
* **Add SMAP** (root-zone moisture at sowing, 2015 on) and **MODIS NDVI** (season peak greenness) as features; both need Earthdata.
* **Drier or hotter futures:** shift the replayed weather (e.g. rain × 0.8) in section 6 to get scenario risk profiles.
* **Rotation** is not in this model: SIAP has no previous-crop information.